In [ ]:
# ================== 0) General Setup + Colab Drive ==================
import os, random, pickle, itertools
from datetime import datetime
import numpy as np

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ================== 1) PyTorch & TorchVision ==================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset

print(f"[INFO] Started at {datetime.now().isoformat()}")

# ================== 2) CUDA & Seeds ==================
print("[INFO] torch.cuda.is_available() ->", torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        _x = torch.zeros(1, device='cuda'); print("[INFO] CUDA sanity OK on device:", _x.device)
    except Exception as e:
        raise SystemExit("[FATAL] CUDA sanity failed: " + repr(e))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("[INFO] Using device:", device)
if device.type == 'cuda':
    print("[INFO] CUDA device:", torch.cuda.get_device_name(0))
    print("[INFO] CUDA capability:", torch.cuda.get_device_capability(0))
    print("[INFO] CUDA current mem (MB):", torch.cuda.memory_reserved() / (1024**2))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if device.type == 'cuda': torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"[INFO] Seeds set to {SEED}")

# ================== 3) Paths ==================
model_path      = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gtask1_best_test_for_finetune.pth"
topk_path       = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gfisher_task1F_cifar10_topk.pkl"
neighbors_path  = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gfisher_task1F_cifar10_neighbors.pkl"
ckpt_dir        = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark"
os.makedirs(ckpt_dir, exist_ok=True)

def _check_file(path, tag):
    if not os.path.exists(path): raise FileNotFoundError(f"[MISSING] {tag}: {path}")
    print(f"[OK] {tag} exists ({os.path.getsize(path)/(1024**2):.2f} MB): {path}")

_check_file(model_path, "Checkpoint(task1)")
_check_file(topk_path, "Fisher Top-K")
_check_file(neighbors_path, "Fisher Neighbors")

# ================== 4) CIFAR-10 Data ==================
tf_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),        # Augmentation
    transforms.RandomHorizontalFlip(),           # Augmentation
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)),
])
tf_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616)),
])

print("[INFO] Loading CIFAR-10 ...")
train_set = datasets.CIFAR10(root="./data", train=True,  download=True, transform=tf_train)
test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=tf_test)
print(f"[INFO] Train size={len(train_set)} | Test size={len(test_set)}")

def indices_for_classes(dataset, keep):
    t = dataset.targets if hasattr(dataset, "targets") else [dataset[i][1] for i in range(len(dataset))]
    return [i for i, y in enumerate(t) if int(y) in keep]

idx_tr_23 = indices_for_classes(train_set, [2,3])  # Task 2 training
idx_te_23 = indices_for_classes(test_set,  [2,3])   # Task 2 testing
idx_te_01 = indices_for_classes(test_set,  [0,1])   # Task 1 testing

class RemapDataset(Dataset):
    def __init__(self, dataset, indices, keep_classes):
        self.dataset = dataset
        self.indices = list(indices)
        self.keep = sorted(int(c) for c in keep_classes)
        assert len(self.keep)==2
        self.mapping = {c:i for i,c in enumerate(self.keep)}
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        orig_idx = self.indices[idx]
        x, y = self.dataset[orig_idx]
        return x, self.mapping[int(y)]

train_23_full = RemapDataset(train_set, idx_tr_23, [2,3])
test_23       = RemapDataset(test_set,  idx_te_23, [2,3])
test_01       = RemapDataset(test_set,  idx_te_01, [0,1])

def make_loader(ds, bs, shuffle, seed=SEED, num_workers=2):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g,
                      num_workers=num_workers, pin_memory=(device.type=='cuda'),
                      persistent_workers=(num_workers>0))


# ================== 5) Architecture ==================
logsoft = nn.LogSoftmax(dim=1)

GN_GROUPS = 32
def make_gn(C: int) -> nn.GroupNorm:
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, 3, stride, 1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1   = make_gn(planes)
        self.conv2 = conv3x3(planes, planes, 1)
        self.gn2   = make_gn(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride, bias=False),
                make_gn(planes)
            )
    def forward(self, x):
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        return torch.relu(out)

class ResNet18Backbone(nn.Module):
    def __init__(self, nf=64):
        super().__init__()
        self.nf = nf
        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)
        self.layer1 = nn.Sequential(BasicBlock(nf, nf, 1), BasicBlock(nf, nf, 1))
        self.layer2 = nn.Sequential(BasicBlock(nf, nf*2, 2), BasicBlock(nf*2, nf*2, 1))
        self.layer3 = nn.Sequential(BasicBlock(nf*2, nf*4, 2), BasicBlock(nf*4, nf*4, 1))
        self.layer4 = nn.Sequential(BasicBlock(nf*4, nf*8, 2), BasicBlock(nf*8, nf*8, 1))
    def forward(self, x):
        x = torch.relu(self.gn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        x = torch.nn.functional.avg_pool2d(x, x.shape[2]); x = x.view(x.size(0), -1)
        return x
    @property
    def out_dim(self): return self.nf*8

class MultiHeadNet(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleDict()
    def add_head(self, name, num_classes):
        head = nn.Linear(self.backbone.out_dim, num_classes)
        head = head.to(next(self.backbone.parameters()).device)
        self.heads[name] = head
    def forward(self, x, head):
        feat = self.backbone(x)
        return self.heads[head](feat)

# ================== 6) Top-K & Neighbors ==================
with open(topk_path, "rb") as f: topk_fisher = pickle.load(f)
with open(neighbors_path, "rb") as f: fisher_neighbors = pickle.load(f)
TOPK_COUNT = len(topk_fisher)
NEIGH_COUNT = len(fisher_neighbors)

# ================== 7) EWC & Freeze Helpers ==================
def build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values):
    pm = dict(model.named_parameters())
    usable = [n for n in fisher_neighbors if n['name'] in pm]
    if not usable: return {}
    max_f = max((n['fisher'] for n in usable), default=1.0) or 1.0
    buckets = {}
    for n in usable:
        buckets.setdefault(n['name'], []).append((int(n['index']), float(n['fisher'])/max_f))
    ewc = {}
    for name, lst in buckets.items():
        lst.sort(key=lambda t:t[0])
        idxs = torch.tensor([i for i,_ in lst], device=device, dtype=torch.long)
        fish = torch.tensor([f for _,f in lst], device=device, dtype=torch.float32)
        flat = pm[name].view(-1)
        orig = torch.stack([neighbor_original_values[(name, int(i))] for i in idxs.tolist()]).to(flat.device, dtype=flat.dtype)
        ewc[name] = {'idxs': idxs, 'fish': fish, 'orig': orig}
    return ewc

def build_freeze_masks_and_cache(model, topk_list):
    """
    Freeze all Top-K parameters except:
      - The current task head: heads.task2.
    """
    masks, frozen_idxs, frozen_vals = {}, {}, {}
    pm = dict(model.named_parameters())

    by_name = {}
    for e in topk_list:
        n, i = e['name'], int(e['index'])
        is_bn = (".bn" in n) or ("bn" in n.lower())
        if (n in pm) and (not n.startswith("heads.task2.")) and (not is_bn):
            by_name.setdefault(n, []).append(i)

    for name, idxs in by_name.items():
        p = pm[name]
        flat = p.detach().view(-1)
        idxs_t = torch.tensor(idxs, device=flat.device, dtype=torch.long)

        if p.requires_grad:
            m = torch.ones_like(p, dtype=torch.bool, device=p.device)
            mv = m.view(-1); mv[idxs_t] = False
            masks[name] = mv.view_as(m)

        with torch.no_grad():
            frozen_idxs[name] = idxs_t
            frozen_vals[name] = flat.index_select(0, idxs_t).clone()

    return masks, frozen_idxs, frozen_vals

def apply_freeze_after_backward(model, masks):
    with torch.no_grad():
        for n,p in model.named_parameters():
            m = masks.get(n, None)
            if p.grad is not None and m is not None:
                p.grad.mul_(m.to(p.grad.dtype))

@torch.no_grad()
def apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals):
    for n, idxs in frozen_idxs.items():
        if n in param_map:
            flat = param_map[n].view(-1)
            flat.index_copy_(0, idxs, frozen_vals[n].to(flat.device, dtype=flat.dtype))

def mask_stats(masks):
    total = sum(m.numel() for m in masks.values())
    frozen = sum((~m).sum().item() for m in masks.values())
    return total, frozen

def audit_topk_vs_masks(model, topk_list):
    pm = dict(model.named_parameters())
    unique_pairs = set((e['name'], int(e['index'])) for e in topk_list)
    excl_not_found = excl_head_t2 = excl_no_grad = excl_oob = 0
    included = set()
    for name, idx in unique_pairs:
        p = pm.get(name, None)
        if p is None:
            excl_not_found += 1; continue
        if name.startswith("heads.task2."):
            excl_head_t2 += 1; continue
        if idx < 0 or idx >= p.numel():
            excl_oob += 1; continue
        if not p.requires_grad:
            excl_no_grad += 1
        included.add((name, idx))
    print(f"[AUDIT] TopK unique pairs     : {len(unique_pairs)}")
    print(f"[AUDIT] Excluded heads.task2.*: {excl_head_t2}")
    print(f"[AUDIT] Not found             : {excl_not_found}")
    print(f"[AUDIT] Out-of-bounds         : {excl_oob}")
    print(f"[AUDIT] No-grad params        : {excl_no_grad}")
    print(f"[AUDIT] Will be masked/strict : {len(included)}")

def freeze_backbone_bn_running_stats(model):
    for m in model.backbone.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()

# ================== 8) Model and Regularizer Initialization ==================
def init_model_and_regularizers(lambda_ewc):
    backbone = ResNet18Backbone(nf=64).to(device)
    model = MultiHeadNet(backbone).to(device)
    for t in ["task1","task2","task3","task4","task5"]:
        model.add_head(t, 2)
    model.to(device)

    ckpt = torch.load(model_path, map_location=device)
    sd = ckpt["model_state"] if "model_state" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:    print("[INIT] Missing keys:", missing)
    if unexpected: print("[INIT] Unexpected keys:", unexpected)

    # (1) Freeze the Task 1 head while training Task 2
    for p in model.heads["task1"].parameters():
        p.requires_grad = False

    # (2) Copy Task 1 head weights to Task 2 head (warm start)
    with torch.no_grad():
        if ("task1" in model.heads) and ("task2" in model.heads):
            h1 = model.heads["task1"]; h2 = model.heads["task2"]
            same_W = (h1.weight.shape == h2.weight.shape)
            same_b = (h1.bias is not None) and (h2.bias is not None) and (h1.bias.shape == h2.bias.shape)
            if same_W: h2.weight.copy_(h1.weight)
            if same_b: h2.bias.copy_(h1.bias)
            print(f"[INIT] Copied task1 → task2 head | W={same_W} | b={same_b}")
        else:
            print("[INIT] WARN: task1/task2 head not found — skip head weight copy")

    # (3) Prepare EWC reference values
    with torch.no_grad():
        cpu_cache = {n: p.view(-1).detach().cpu() for n,p in model.named_parameters()}

    neighbor_original_values = {}
    for n in fisher_neighbors:
        name, idx = n['name'], int(n['index'])
        if name in cpu_cache and idx < cpu_cache[name].numel():
            neighbor_original_values[(name, idx)] = cpu_cache[name][idx]

    ewc_tensors = build_neighbor_tensors(model, fisher_neighbors, neighbor_original_values)

    # (4) Build Top-K freezing masks
    masks, frozen_idxs, frozen_vals = build_freeze_masks_and_cache(model, topk_fisher)

    audit_topk_vs_masks(model, topk_fisher)
    return model, ewc_tensors, masks, frozen_idxs, frozen_vals

# ================== 9) Evaluation ==================
@torch.no_grad()
def evaluate_head(model, loader, head):
    model.eval(); correct=0; total=0
    for x,y in loader:
        x = x.to(device); y = torch.as_tensor(y, device=device, dtype=torch.long)
        logits = model(x, head=head); pred = logits.argmax(1)
        correct += (pred==y).sum().item(); total += y.size(0)
    return 100.0*correct/max(1,total)

# ================== 10) Train One Setting ==================
def train_one_setting(lr_backbone, lr_head, bs, lambda_ewc, epochs):
    model, ewc_tensors, masks, frozen_idxs, frozen_vals = init_model_and_regularizers(lambda_ewc)
    # Split optimizer parameters into four groups:
    # - Backbone with weight decay
    # - Backbone without weight decay (bias)
    # - Task 2 head with weight decay
    # - Task 2 head without weight decay (bias)
    bb_decay, bb_nodecay, hd_decay, hd_nodecay = [], [], [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_head2 = n.startswith("heads.task2.")
        no_decay = (p.dim()==1) or n.endswith(".bias") or ("bn" in n.lower())
        if is_head2:
            (hd_nodecay if no_decay else hd_decay).append(p)
        else:
            (bb_nodecay if no_decay else bb_decay).append(p)

    optimizer = optim.SGD(
        [
            {"params": bb_decay,   "lr": lr_backbone, "weight_decay": 1e-4},
            {"params": bb_nodecay, "lr": lr_backbone, "weight_decay": 0.0},
            {"params": hd_decay,   "lr": lr_head,     "weight_decay": 1e-4},
            {"params": hd_nodecay, "lr": lr_head,     "weight_decay": 0.0},
        ],
        momentum=0.9
    )

    param_map = {n:p for n,p in model.named_parameters()}

    train_loader = make_loader(train_23_full, bs=bs, shuffle=True)
    test1_loader = make_loader(test_01,       bs=256, shuffle=False)
    test2_loader = make_loader(test_23,       bs=256, shuffle=False)

    pre_t1 = evaluate_head(model, test1_loader, head="task1")
    print(f"[PRE] Task1 accuracy BEFORE fine-tuning Task2: {pre_t1:.2f}%")

    total_mask_elems, total_frozen = mask_stats(masks)
    print(f"[INFO]   mask_elems={total_mask_elems} | frozen(TopK)={total_frozen}")
    print(f"[OPT ]   lr_backbone={lr_backbone} | lr_head={lr_head} | "
          f"bb_decay={len(bb_decay)} bb_nodecay={len(bb_nodecay)} | "
          f"hd_decay={len(hd_decay)} hd_nodecay={len(hd_nodecay)}")

    best = {'epoch': -1, 't1': -1.0, 't2': -1.0, 'avg': -1.0, 'model_state': None}
    for e in range(1, epochs+1):
        model.train()
        freeze_backbone_bn_running_stats(model)

        running_loss=0.0; steps=0
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = torch.as_tensor(labels, device=device, dtype=torch.long)
            optimizer.zero_grad(set_to_none=True)

            # EWC penalty (FP32)
            ewc_penalty = 0.0
            if lambda_ewc != 0 and len(ewc_tensors) > 0:
                for name, pack in ewc_tensors.items():
                    p = param_map[name].view(-1)
                    diff = p.index_select(0, pack['idxs']) - pack['orig']
                    ewc_penalty += (pack['fish'] * (diff**2)).sum()

            logits = model(imgs, head="task2")
            loss = nn.functional.cross_entropy(logits, labels) + (lambda_ewc/2.0)*ewc_penalty

            loss.backward()
            apply_freeze_after_backward(model, masks)
            optimizer.step()
            apply_strict_freeze_after_step(param_map, frozen_idxs, frozen_vals)

            running_loss += float(loss.detach().cpu()); steps += 1

        acc_t1 = evaluate_head(model, test1_loader, head="task1")
        acc_t2 = evaluate_head(model, test2_loader, head="task2")
        avg = 0.5*(acc_t1 + acc_t2)
        print(f"  Epoch {e}/{epochs} | train_loss={running_loss/max(1,steps):.4f} | "
              f"T1={acc_t1:.2f}% T2={acc_t2:.2f}% AVG={avg:.2f}%")

        if avg > best['avg'] or (avg == best['avg'] and acc_t2 > best['t2']):
            best = {'epoch': e, 't1': acc_t1, 't2': acc_t2, 'avg': avg,
                    'model_state': {k:v.detach().cpu() for k,v in model.state_dict().items()}}

    return best

# ================== 11) Grid Search ==================
def run_grid_search_and_save_best():
    backbone_lrs = [0.0002]
    head_lrs     = [0.7]
    batch_sizes  = [32]
    lambda_values  = [2]
    epoch_counts   = [60]

    print(f"[DATA ] Train(T2)={len(train_23_full)} | Test(T2)={len(test_23)} | Test(T1)={len(test_01)}")
    print(f"[META ] TopK={TOPK_COUNT} | Neighbors={NEIGH_COUNT}")

    global_best = {'avg': -1.0, 't1': -1.0, 't2': -1.0, 'epoch': -1, 'setting': None, 'state': None}

    for lr_bb, lr_hd, bs, lam, ep in itertools.product(backbone_lrs, head_lrs, batch_sizes, lambda_values, epoch_counts):
        print(f"\n[SETTING] LR_backbone={lr_bb}, LR_head={lr_hd}, BS={bs}, λ={lam}, EPOCHS={ep} "
              f"| TopK={TOPK_COUNT}, Neigh={NEIGH_COUNT}")
        best = train_one_setting(lr_backbone=lr_bb, lr_head=lr_hd, bs=bs, lambda_ewc=lam, epochs=ep)
        print(f"[SETTING-BEST] epoch {best['epoch']} | T1={best['t1']:.2f}% | T2={best['t2']:.2f}% | AVG={best['avg']:.2f}%")
        if best['avg'] > global_best['avg']:
            global_best = {
                'avg': best['avg'], 't1': best['t1'], 't2': best['t2'],
                'epoch': best['epoch'],
                'setting': f"LR_backbone={lr_bb}, LR_head={lr_hd}, BS={bs}, λ={lam}, EPOCHS={ep}",
                'state': best['model_state']
            }

    # Save the best global result

    best_test_path = os.path.join(ckpt_dir, "Gtask2_best_test_for_finetune.pth")
    torch.save({
        "model_state": global_best['state'],      # Best-test weights (state_dict)
        "best_config": global_best['setting'],
        "best_test_acc": global_best['t2'],
        "best_test_epoch": global_best['epoch'],
        "trained_head": "task2",
        "all_heads": ["task1", "task2", "task3", "task4", "task5"],
    }, best_test_path)

    print("\n====================")
    print(f"[GLOBAL BEST] AVG={global_best['avg']:.2f}% | T1={global_best['t1']:.2f}% | T2={global_best['t2']:.2f}% | at epoch {global_best['epoch']}")
    print(f"[SETTING     ] {global_best['setting']}")
    print(f"[SAVED       ] {best_test_path}")


# ================== 12) Run ==================
run_grid_search_and_save_best()


Mounted at /content/drive
[INFO] Started at 2026-08-30T02:48:33.086571
[INFO] torch.cuda.is_available() -> True
[INFO] CUDA sanity OK on device: cuda:0
[INFO] Using device: cuda
[INFO] CUDA device: NVIDIA A100-SXM4-40GB
[INFO] CUDA capability: (8, 0)
[INFO] CUDA current mem (MB): 2.0
[INFO] Seeds set to 42
[OK] Checkpoint(task1) exists (42.65 MB): /content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gtask1_best_test_for_finetune.pth
[OK] Fisher Top-K exists (0.36 MB): /content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gfisher_task1F_cifar10_topk.pkl
[OK] Fisher Neighbors exists (0.22 MB): /content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gfisher_task1F_cifar10_neighbors.pkl
[INFO] Loading CIFAR-10 ...


100%|██████████| 170M/170M [28:31<00:00, 99.6kB/s]


[INFO] Train size=50000 | Test size=10000
[DATA ] Train(T2)=10000 | Test(T2)=2000 | Test(T1)=2000
[META ] TopK=10000 | Neighbors=6162

[SETTING] LR_backbone=0.0002, LR_head=0.7, BS=32, λ=2, EPOCHS=60 | TopK=10000, Neigh=6162
[INIT] Copied task1 → task2 head | W=True | b=True
[AUDIT] TopK unique pairs     : 10000
[AUDIT] Excluded heads.task2.*: 0
[AUDIT] Not found             : 0
[AUDIT] Out-of-bounds         : 0
[AUDIT] No-grad params        : 30
[AUDIT] Will be masked/strict : 10000
[PRE] Task1 accuracy BEFORE fine-tuning Task2: 98.60%
[INFO]   mask_elems=11168320 | frozen(TopK)=9970
[OPT ]   lr_backbone=0.0002 | lr_head=0.7 | bb_decay=23 bb_nodecay=43 | hd_decay=1 hd_nodecay=1
  Epoch 1/60 | train_loss=0.7876 | T1=97.90% T2=64.20% AVG=81.05%
  Epoch 2/60 | train_loss=0.6879 | T1=97.25% T2=66.80% AVG=82.03%
  Epoch 3/60 | train_loss=0.6116 | T1=97.20% T2=65.35% AVG=81.28%
  Epoch 4/60 | train_loss=0.5877 | T1=96.75% T2=70.95% AVG=83.85%
  Epoch 5/60 | train_loss=0.5550 | T1=96.55% T2=